# Comparaison d'algos en Classification Supervisée

Nous allons comparer les différentes méthodes de classification supervisée n que nous avons vues jusqu'à présent, la régression logistique avec toutes les variables, puis  avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC puis les méthodes de vraisemblance pénalisée.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [21]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
import sklearn.metrics as sklm
from sklearn.metrics import log_loss
from patsy import dmatrix

import logistic_step_sk as lss 

In [22]:
don = pd.read_csv("dfbase.csv",header=0,sep=",")
don.head(3)

,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present,Y
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0,1
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0,1
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0,0


Codage des variables qualitatives famhist

In [23]:
X = don.drop(columns=["Y"])
X = X.to_numpy()
Y = don["Y"].to_numpy()

In [24]:
nb=10
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=123)
PROB = pd.DataFrame({"Y":Y,"log":0.0,"BIC":0.0,"AIC":0.0,
                    "ridge":0.0,"lasso":0.0,"elast":0.0,"arbre":0.0,"foret":0.0})

choix des grilles de régularisation

In [25]:
def grille(X, y, type = "lasso", ng=100):
    scalerX = StandardScaler().fit(X)
    Xcr= scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y-y.mean()))).max()/X.shape[0]
    llc = np.linspace(0,-4,ng)
    ll = l0*10**llc
    if type=="lasso":
        Cs = 1/ 0.9/ X.shape[0] / (l0*10**(llc))
    elif type=="ridge":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 100)
    elif type=="enet":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 2)
    return Cs

On compare les méthodes en utilisant skf

In [26]:
for app_index, val_index in skf.split(X,Y):
    Xapp = X[app_index,:]
    Xtest = X[val_index,:]
    Yapp = Y[app_index]
    ### logistique
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="bic",multi_class='auto').fit(Xapp,Yapp)
    PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="aic",multi_class='auto').fit(Xapp,Yapp)
    PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2",Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]
    ###arbre
    arbre = DecisionTreeClassifier(min_samples_leaf=5).fit(Xapp,Yapp)
    PROB.loc[val_index,"arbre"] = arbre.predict_proba(Xtest)[:,1]
    ###foret
    foret = RandomForestClassifier().fit(Xapp,Yapp)
    PROB.loc[val_index,"foret"] = foret.predict_proba(Xtest)[:,1]
    ###gradient boosting
    params = dict(n_estimators=1000, max_depth=1, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10) # important n_iter_no_change à fixer pour early stopping !!!
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbm1"] = gbm_early_stopping.predict_proba(Xtest)[:, 1]
    params = dict(n_estimators=1000, max_depth=2, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbm2"] = gbm_early_stopping.predict_proba(Xtest)[:, 1]
    ###stochastic gradient boosting - seulement si n très grand - à documenter
    params = dict(n_estimators=5000, max_depth=1, learning_rate=0.1, random_state=42,subsample=0.2)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbms1"] = gbm_early_stopping.predict_proba(Xtest)[:, 1]
    params = dict(n_estimators=5000, max_depth=2, learning_rate=0.1, random_state=42, subsample=0.2)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbms2"] = gbm_early_stopping.predict_proba(Xtest)[:, 1]
    ###adaboost 
    # séparation train / validation
    X_train, X_val, y_train, y_val = train_test_split(Xapp, Yapp, test_size=0.1, random_state=42, stratify=Yapp)
    # AdaBoost depth = 1
    base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
    ada = AdaBoostClassifier(estimator=base_tree, n_estimators=5000, learning_rate=0.1, random_state=42)
    ada.fit(X_train, y_train)

best_iter = 1
best_loss = np.inf
patience = 10
counter = 0

# staged_predict_proba permet de suivre chaque itération
for i, y_proba in enumerate(ada.staged_predict_proba(X_val), start=1):

    loss = log_loss(y_val, y_proba)

    if loss < best_loss:
        best_loss = loss
        best_iter = i
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        break

print("Best iteration:", best_iter)

# réentraînement avec le meilleur nombre d'arbres
ada_final = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=best_iter,
    learning_rate=0.1,
    random_state=42
)

ada_final.fit(Xapp, Yapp)

PROB.loc[val_index, "ada1"] = ada_final.predict_proba(Xtest)[:, 1]

############################################
# AdaBoost depth = 2
############################################

base_tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

ada = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=5000,
    learning_rate=0.1,
    random_state=42
)

ada.fit(X_train, y_train)

best_iter = 1
best_loss = np.inf
counter = 0

for i, y_proba in enumerate(ada.staged_predict_proba(X_val), start=1):

    loss = log_loss(y_val, y_proba)

    if loss < best_loss:
        best_loss = loss
        best_iter = i
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        break

print("Best iteration:", best_iter)

ada_final = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=best_iter,
    learning_rate=0.1,
    random_state=42
)

ada_final.fit(Xapp, Yapp)

PROB.loc[val_index, "ada2"] = ada_final.predict_proba(Xtest)[:, 1]


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

63
24
22
22


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

92
48
33
33


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

...

190
44
41
37


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

92
87
30
36


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

38
12
12
22
Best iteration: 40
Best iteration: 70


In [27]:
round(PROB.iloc[0:4,:],3)

,Y,log,BIC,AIC,ridge,lasso,elast,arbre,foret,gbm1,gbm2,gbms1,gbms2,ada1,ada2
0,1,0.742,0.689,0.689,0.677,0.598,0.605,1.000,0.76,0.624,0.769,0.643,0.845,NaN,NaN
1,1,0.292,0.362,0.333,0.317,0.373,0.358,0.000,0.31,0.294,0.359,0.427,0.485,0.316,0.274
2,0,0.251,0.313,0.230,0.286,0.314,0.297,0.667,0.14,0.201,0.166,0.175,0.097,NaN,NaN
3,1,0.719,0.720,0.681,0.679,0.696,0.682,1.000,0.69,0.657,0.753,0.667,0.499,NaN,NaN


In [28]:
PROB.to_csv("PROB.csv",index=False)

In [29]:
PROB.to_csv("PROBpoly.csv",index=False)

In [36]:
PROB.to_csv("PROBinter.csv",index=False)